# 🌸 Mythri — Sarvam-30B QLoRA Fine-Tuning
## Behavioral Therapeutic Companion for Indian Languages

**Model**: `sarvamai/sarvam-30b`  
**Method**: QLoRA (4-bit NF4 + bfloat16)  
**Dataset**: 60 multilingual behavioral therapeutic conversations  
**Languages**: Hindi, Telugu, Tamil, Kannada, Bengali, Punjabi, Marathi, Hinglish, English  

---
**Runtime**: Use `Runtime → Change runtime type → A100 GPU`


In [ ]:
# Step 1: Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Step 2: Install dependencies
!pip install -q transformers==4.44.2 peft==0.12.0 trl==0.11.4 bitsandbytes==0.43.1 \
    datasets==2.21.0 accelerate==0.34.2 evaluate rouge-score bert-score sacrebleu
print('✅ Dependencies installed')

In [ ]:
# Step 3: Mount Google Drive (to save model)
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/mythri-sarvam-30b-adapter', exist_ok=True)
print('✅ Drive mounted. Adapter will be saved to Google Drive.')

In [ ]:
# Step 4: Upload your dataset files
# Upload train.jsonl and eval.jsonl from: backend/finetuning/data/
from google.colab import files

print('Upload train.jsonl:')
uploaded = files.upload()
print('Upload eval.jsonl:')
uploaded = files.upload()

import os
os.makedirs('/content/data', exist_ok=True)
# Move uploaded files
!mv train.jsonl /content/data/train.jsonl 2>/dev/null || echo 'train.jsonl already in place'
!mv eval.jsonl /content/data/eval.jsonl 2>/dev/null || echo 'eval.jsonl already in place'

# Verify
!wc -l /content/data/train.jsonl
!wc -l /content/data/eval.jsonl
print('✅ Dataset ready')

In [ ]:
# Step 5: Configuration
import torch

CONFIG = {
    'base_model': 'sarvamai/sarvam-30b',
    'max_seq_length': 2048,
    'num_train_epochs': 3,
    'per_device_train_batch_size': 2,
    'per_device_eval_batch_size': 2,
    'gradient_accumulation_steps': 4,  # effective batch = 8
    'learning_rate': 1e-4,
    'weight_decay': 0.01,
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'logging_steps': 5,
    'eval_steps': 25,
    'save_steps': 50,
    'optim': 'paged_adamw_8bit',
    # LoRA
    'lora_r': 64,
    'lora_alpha': 128,
    'lora_dropout': 0.05,
    'lora_target_modules': [
        'q_proj', 'v_proj', 'k_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    # 4-bit
    'load_in_4bit': True,
    'bnb_4bit_quant_type': 'nf4',
    'bnb_4bit_compute_dtype': 'bfloat16',
    'bnb_4bit_use_double_quant': True,
}

OUTPUT_DIR = '/content/drive/MyDrive/mythri-sarvam-30b-adapter'
print('✅ Config set. Base model:', CONFIG['base_model'])

In [ ]:
# Step 6: Load tokenizer
from transformers import AutoTokenizer

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG['base_model'],
    trust_remote_code=True,
    padding_side='right'
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('✅ Tokenizer loaded')

In [ ]:
# Step 7: Load model in 4-bit
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {CONFIG["base_model"]} in 4-bit...')
print('This may take 5-10 minutes on first run (downloads ~16GB)...')
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['base_model'],
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
print('✅ Model loaded in 4-bit')

In [ ]:
# Step 8: Attach LoRA adapters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    target_modules=CONFIG['lora_target_modules'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA attached: {trainable_params:,} / {total_params:,} params ({100*trainable_params/total_params:.2f}% trainable)')

In [ ]:
# Step 9: Load & format dataset
from datasets import load_dataset

SYSTEM_PROMPT = (
    'Aap Mythri ho — ek gehri samajh rakhne waali, warmth se bhari ek dost. '
    'Aap CBT, DBT, ACT, aur psychodynamic therapy mein trained ho, lekin kabhi bhi '
    'textbook ki tarah nahi bolte. Aap samne wale ki baat sunti ho, unke emotions validate '
    'karti ho, aur phir dheere dheere unhe apne thoughts aur feelings ko samajhne mein madad '
    'karti ho. Indian cultural context aapko deeply pata hai. '
    'Aap uss language mein respond karti ho jismein user baat karta hai.'
)

def format_example(example):
    text = ''
    for msg in example['messages']:
        role = msg['role']
        content = msg['content']
        if role == 'system':
            text += f'<|system|>\n{content}\n'
        elif role == 'user':
            text += f'<|user|>\n{content}\n'
        elif role == 'assistant':
            text += f'<|assistant|>\n{content}\n'
    text += tokenizer.eos_token or '</s>'
    return {'text': text}

raw_ds = load_dataset('json', data_files={
    'train': '/content/data/train.jsonl',
    'eval': '/content/data/eval.jsonl'
})

dataset = raw_ds.map(
    format_example,
    remove_columns=raw_ds['train'].column_names
)

print(f'✅ Dataset formatted: {len(dataset["train"])} train, {len(dataset["eval"])} eval')
print('Sample:', dataset['train'][0]['text'][:200])

In [ ]:
# Step 10: Train!
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_eval_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    warmup_ratio=CONFIG['warmup_ratio'],
    lr_scheduler_type=CONFIG['lr_scheduler_type'],
    logging_steps=CONFIG['logging_steps'],
    eval_strategy='steps',
    eval_steps=CONFIG['eval_steps'],
    save_steps=CONFIG['save_steps'],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    optim=CONFIG['optim'],
    bf16=True,
    fp16=False,
    report_to='none',
    remove_unused_columns=False,
    dataset_text_field='text',
    max_length=CONFIG['max_seq_length'],
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['eval'],
)

print('🚀 Starting training...')
trainer.train()

# Save final adapter
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✅ Training complete! Adapter saved to: {OUTPUT_DIR}')

In [ ]:
# Step 11: Quick inference test
from peft import PeftModel
from transformers import pipeline

test_questions = [
    'Main bahut anxious rehta hoon, kuch samajh nahi aata.',  # Hindi
    'నాకు చాలా stress గా ఉంది.',  # Telugu
    'I feel so lost and hopeless lately.',  # English
    'Yaar, sab kuch bahut overwhelming lag raha hai.',  # Hinglish
]

for question in test_questions:
    print('\n' + '='*60)
    print('User:', question)
    prompt = f'<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{question}\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print('Mythri:', response)